# Step 3: run the open models on Colab

Runtime → Change runtime type → **T4 GPU**. Run the cells top to bottom. Each run appends to `outputs/raw/<model>/`, so if Colab disconnects, run the same cell again and it resumes.

Order: do the dry run first and check the printed outputs before the full runs.

In [1]:
!git clone https://github.com/shuaige387-a11y/LLM_Wug-Test_on_Dutch_Diminutives.git repo
%cd repo
!pip install -q -r requirements-gpu.txt

Cloning into 'repo'...
remote: Enumerating objects: 78, done.
remote: Counting objects: 100% (78/78), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 78 (delta 16), reused 68 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (78/78), 54.66 KiB | 6.07 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/repo


Some models (Gemma) ask you to accept their license on Hugging Face first. Accept it on the model page while logged in, then paste a read token here.

In [2]:
from huggingface_hub import notebook_login
notebook_login()

## Dry run: 3 words, one small model
Check that the scores look sane and the generated answers are short Dutch words.

In [3]:
!python scripts/03_run_hf.py --model eurollm-1.7b --method B --limit 3
!python scripts/03_run_hf.py --model eurollm-1.7b-instruct --method A --limit 3

eurollm-1.7b method B: 30 items, 0 already done, 30 to run
config.json: 100% 637/637 [00:00<00:00, 2.91MB/s]
tokenizer_config.json: 100% 960/960 [00:00<00:00, 4.46MB/s]
tokenizer.json: 100% 8.18M/8.18M [00:00<00:00, 19.6MB/s]

tokenizer.model: downloading bytes:   0% 0.00/2.41M [00:00<?, ?B/s]
tokenizer.model: downloading bytes: 100% 1.75M/1.75M [00:01<00:00, 1.26MB/s,  169kB/s  ]
tokenizer.model: reconstructing file: 100% 2.41M/2.41M [00:01<00:00, 1.73MB/s,  232kB/s  ]
special_tokens_map.json: 100% 414/414 [00:00<00:00, 1.78MB/s]

pytorch_model.bin: downloading bytes:   0% 0.00/3.31G [00:00<?, ?B/s]
pytorch_model.bin: downloading bytes:   0% 13.7M/3.31G [00:01<05:01, 11.0MB/s,  706kB/s  ]
pytorch_model.bin: downloading bytes:   1% 24.1M/3.31G [00:02<02:27, 22.3MB/s, 1.30MB/s  ]
pytorch_model.bin: downloading bytes:   2% 50.9M/3.31G [00:02<01:02, 52.1MB/s, 3.88MB/s  ]
pytorch_model.bin: reconstructing file:   1% 37.8M/3.31G [00:02<02:00, 27.3MB/s, 1.88MB/s  ]
pytorch_model.bin: downloa

In [5]:
import pandas as pd
s = pd.read_json('outputs/raw/eurollm-1.7b/scoring.jsonl', lines=True)
display(s[s.context=='frame'][['stem','candidate_form','is_correct','logprob_total','n_tokens','tokens']])
g = pd.read_json('outputs/raw/eurollm-1.7b-instruct/generation.jsonl', lines=True)
display(g[g.decoding=='greedy'][['stem','template','output']])

,stem,candidate_form,is_correct,logprob_total,n_tokens,tokens
0,tuif,tuifje,True,-3.394018,4,"[▁tu, if, je, .]"
2,tuif,tuiftje,False,-8.437604,4,"[▁tu, ift, je, .]"
4,tuif,tuifpje,False,-15.282042,5,"[▁tu, if, p, je, .]"
6,tuif,tuifkje,False,-19.848204,5,"[▁tu, if, k, je, .]"
8,tuif,tuifetje,False,-12.385097,4,"[▁tu, if, etje, .]"
10,bauk,baukje,True,-6.003998,4,"[▁b, auk, je, .]"
12,bauk,bauktje,False,-12.742534,5,"[▁b, au, kt, je, .]"
14,bauk,baukpje,False,-16.103202,5,"[▁b, auk, p, je, .]"
16,bauk,baukkje,False,-14.006144,5,"[▁b, au, kk, je, .]"
18,bauk,bauketje,False,-14.573059,5,"[▁b, au, ket, je, .]"


,stem,template,output
0,tuif,T1_original,Object 4 is een tuif.
11,tuif,T2_frame,Dit is een tuif.
22,tuif,T3_two_objects,Het kleine voorwerp is een tuif.
33,tuif,T4_metalinguistic,Het verkleinwoord van 'tuif' is 'tuifje'.
44,tuif,T5_fewshot,Het verkleinwoord van 'tuif' is 'tuifje'.
55,bauk,T1_original,Object 6 is een verkleinde versie van object 2.
66,bauk,T2_frame,Dit is een huis.
77,bauk,T3_two_objects,Het kleine voorwerp is een bakje.
88,bauk,T4_metalinguistic,Het verkleinwoord van 'bauk' is 'baukje'.
99,bauk,T5_fewshot,Bauwke


Delete the dry-run files before the full run, so the full run starts clean:

In [6]:
!rm -rf outputs/raw/eurollm-1.7b outputs/raw/eurollm-1.7b-instruct

## Full runs
One model per cell. `--words core` = 15 core nonce words + 20 real words; add `--words all` later for the 51 reserve words. Restart the runtime between big models if memory runs out.

In [7]:
for m, meth in [('eurollm-1.7b','B'), ('eurollm-1.7b','A'), ('eurollm-1.7b-instruct','A'), ('fietje-2','B'), ('fietje-2','A')]:
    !python scripts/03_run_hf.py --model {m} --method {meth}

eurollm-1.7b method B: 350 items, 0 already done, 350 to run
Loading weights: 100% 219/219 [00:13<00:00, 16.59it/s]
  50 records
  100 records
  150 records
  200 records
  250 records
  300 records
  350 records
wrote 350 records to outputs/raw/eurollm-1.7b/scoring.jsonl
eurollm-1.7b method A: 175 items, 0 already done, 175 to run
Loading weights: 100% 219/219 [00:01<00:00, 198.34it/s]
  50 records
  100 records
  150 records
  200 records
  250 records
  300 records
  350 records
  400 records
  450 records
  500 records
  550 records
  600 records
  650 records
  700 records
  750 records
  800 records
  850 records
  900 records
  950 records
  1000 records
  1050 records
  1100 records
  1150 records
  1200 records
  1250 records
  1300 records
  1350 records
  1400 records
  1450 records
  1500 records
  1550 records
  1600 records
  1650 records
  1700 records
  1750 records
  1800 records
  1850 records
  1900 records
wrote 1925 records to outputs/raw/eurollm-1.7b/generation.js

In [8]:
for m, meth in [('gemma4-e2b','B'), ('gemma4-e2b-it','A'), ('gemma4-e4b','B'), ('gemma4-e4b-it','A')]:
    !python scripts/03_run_hf.py --model {m} --method {meth}

gemma4-e2b method B: 350 items, 0 already done, 350 to run
config.json: 100% 4.91k/4.91k [00:00<00:00, 1.87MB/s]
tokenizer_config.json: 100% 906/906 [00:00<00:00, 3.97MB/s]

tokenizer.json: downloading bytes:   0% 0.00/32.2M [00:00<?, ?B/s]
tokenizer.json: downloading bytes:  17% 5.34M/32.2M [00:01<00:07, 3.72MB/s,  387kB/s  ]
tokenizer.json: downloading bytes:  25% 7.90M/32.2M [00:01<00:03, 6.21MB/s,  511kB/s  ]
tokenizer.json: downloading bytes: 100% 8.75M/8.75M [00:01<00:00, 4.47MB/s,  827kB/s  ]
tokenizer.json: reconstructing file: 100% 32.2M/32.2M [00:01<00:00, 16.4MB/s, 3.06MB/s  ]

model.safetensors: downloading bytes:   0% 8.05M/10.2G [00:01<28:02, 6.08MB/s]
model.safetensors: downloading bytes:   1% 51.7M/10.2G [00:01<03:16, 51.8MB/s, 3.54MB/s  ]
model.safetensors: reconstructing file:   0% 302k/10.2G [00:01<14:06:16, 202kB/s, 12.6kB/s  ]
model.safetensors: downloading bytes:   1% 92.7M/10.2G [00:01<01:55, 87.8MB/s, 7.27MB/s  ]
model.safetensors: downloading bytes:   1% 119M/1

In [9]:
for m, meth in [('eurollm-9b','B'), ('eurollm-9b-instruct','A'), ('gemma4-12b','B'), ('gemma4-12b-it','A')]:
    !python scripts/03_run_hf.py --model {m} --method {meth}

eurollm-9b method B: 350 items, 0 already done, 350 to run
config.json: 100% 722/722 [00:00<00:00, 1.75MB/s]
tokenizer_config.json: 100% 47.0k/47.0k [00:00<00:00, 11.6MB/s]

tokenizer.json: downloading bytes:   0% 0.00/15.8M [00:00<?, ?B/s]
tokenizer.json: downloading bytes: 100% 4.10M/4.10M [00:01<00:00, 2.64MB/s,  396kB/s  ]
tokenizer.json: reconstructing file: 100% 15.8M/15.8M [00:01<00:00, 10.1MB/s, 1.52MB/s  ]

tokenizer.model: downloading bytes:   0% 0.00/2.41M [00:00<?, ?B/s]
tokenizer.model: downloading bytes: 100% 1.75M/1.75M [00:01<00:00, 1.24MB/s,  169kB/s  ]
tokenizer.model: reconstructing file: 100% 2.41M/2.41M [00:01<00:00, 1.71MB/s,  233kB/s  ]
special_tokens_map.json: 100% 414/414 [00:00<00:00, 1.93MB/s]
model.safetensors.index.json: 100% 31.4k/31.4k [00:00<00:00, 59.1MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0% 0/8 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/4.83G [00:00<?, ?B/s]      

A shrotage of storage occured, and the following codes resolve it.

In [11]:
!ls outputs/raw
!rm -rf ~/.cache/huggingface/hub/models--*
!df -h /content

eurollm-1.7b	       eurollm-9b-instruct  gemma4-12b-it  gemma4-e4b
eurollm-1.7b-instruct  fietje-2		    gemma4-e2b	   gemma4-e4b-it
eurollm-9b	       gemma4-12b	    gemma4-e2b-it
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   98G   15G  87% /


In [1]:
%cd /content/repo
!du -sh ~/.cache/huggingface/* 2>/dev/null
!rm -rf ~/.cache/huggingface/hub ~/.cache/huggingface/xet
!df -h /content

/content/repo
98G	/root/.cache/huggingface/hub
4.0K	/root/.cache/huggingface/stored_tokens
4.0K	/root/.cache/huggingface/token
11M	/root/.cache/huggingface/xet
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G  312M  113G   1% /


In [2]:
from google.colab import files
for m, meth in [('eurollm-9b-instruct','A'), ('gemma4-12b','B'), ('gemma4-12b-it','A')]:
    !python scripts/03_run_hf.py --model {m} --method {meth}
    !rm -rf ~/.cache/huggingface/hub ~/.cache/huggingface/xet
    !zip -qr outputs_raw.zip outputs/raw
    files.download('outputs_raw.zip')

eurollm-9b-instruct method A: 175 items, 0 already done, 175 to run
config.json: 100% 680/680 [00:00<00:00, 3.12MB/s]
tokenizer_config.json: 100% 47.0k/47.0k [00:00<00:00, 79.6MB/s]

tokenizer.json: downloading bytes:  20% 3.15M/15.8M [00:01<00:04, 2.57MB/s]
tokenizer.json: downloading bytes: 100% 4.10M/4.10M [00:01<00:00, 2.77MB/s,  393kB/s  ]
tokenizer.json: reconstructing file: 100% 15.8M/15.8M [00:01<00:00, 10.7MB/s, 1.52MB/s  ]

tokenizer.model: downloading bytes:   0% 0.00/2.41M [00:00<?, ?B/s]
tokenizer.model: downloading bytes: 100% 1.75M/1.75M [00:01<00:00, 1.37MB/s,  169kB/s  ]
tokenizer.model: reconstructing file: 100% 2.41M/2.41M [00:01<00:00, 1.89MB/s,  233kB/s  ]
special_tokens_map.json: 100% 557/557 [00:00<00:00, 1.98MB/s]
chat_template.jinja: 100% 295/295 [00:00<00:00, 1.85MB/s]
model.safetensors.index.json: 100% 31.4k/31.4k [00:00<00:00, 65.4MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0% 0/4 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

gemma4-12b method B: 350 items, 0 already done, 350 to run
config.json: 100% 4.38k/4.38k [00:00<00:00, 10.8MB/s]
tokenizer_config.json: 100% 888/888 [00:00<00:00, 4.48MB/s]

tokenizer.json: downloading bytes:   0% 0.00/32.2M [00:00<?, ?B/s]
tokenizer.json: downloading bytes:  21% 6.68M/32.2M [00:01<00:05, 4.89MB/s,  529kB/s  ]
tokenizer.json: downloading bytes:  26% 8.24M/32.2M [00:01<00:03, 6.19MB/s,  639kB/s  ]
tokenizer.json: downloading bytes: 100% 8.75M/8.75M [00:01<00:00, 4.88MB/s,  824kB/s  ]
tokenizer.json: reconstructing file: 100% 32.2M/32.2M [00:01<00:00, 17.9MB/s, 3.04MB/s  ]

model.safetensors: downloading bytes:   0% 0.00/23.9G [00:00<?, ?B/s]
model.safetensors: downloading bytes:   2% 371M/23.9G [00:02<01:30, 260MB/s, 32.6MB/s  ]
model.safetensors: downloading bytes:   2% 444M/23.9G [00:02<01:53, 207MB/s, 36.5MB/s  ]
model.safetensors: downloading bytes:   2% 553M/23.9G [00:03<01:36, 243MB/s, 45.3MB/s  ]
model.safetensors: downloading bytes:   3% 720M/23.9G [00:03<01:22,

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

gemma4-12b-it method A: 175 items, 0 already done, 175 to run
config.json: 100% 4.42k/4.42k [00:00<00:00, 10.5MB/s]
tokenizer_config.json: 100% 3.09k/3.09k [00:00<00:00, 7.45MB/s]

tokenizer.json: downloading bytes:   2% 581k/32.2M [00:01<01:12, 438kB/s]
tokenizer.json: downloading bytes:  16% 5.20M/32.2M [00:01<00:05, 4.88MB/s, 56.1kB/s  ]
tokenizer.json: downloading bytes: 100% 8.75M/8.75M [00:01<00:00, 4.69MB/s,  830kB/s  ]
tokenizer.json: reconstructing file: 100% 32.2M/32.2M [00:01<00:00, 17.2MB/s, 3.07MB/s  ]
chat_template.jinja: 100% 18.7k/18.7k [00:00<00:00, 35.0MB/s]

model.safetensors: downloading bytes:   0% 0.00/23.9G [00:00<?, ?B/s]
model.safetensors: downloading bytes:   1% 127M/23.9G [00:01<02:35, 153MB/s, 9.30MB/s  ] 
model.safetensors: downloading bytes:   1% 298M/23.9G [00:02<01:40, 235MB/s, 26.3MB/s  ]
model.safetensors: downloading bytes:   2% 553M/23.9G [00:03<01:20, 291MB/s, 45.3MB/s  ]
model.safetensors: downloading bytes:   5% 1.27G/23.9G [00:05<01:15, 300MB/s, 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Save the outputs
Download the zip and upload it to the project, or commit it from here.

In [10]:
!zip -qr outputs_raw.zip outputs/raw
from google.colab import files
files.download('outputs_raw.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>